# 작물 탐지 YOLO 모델 학습 파이프라인 (Curriculum Learning)

이 노트북은 스마트팜 로봇의 객체 판별 AI(4클래스: `eggplant`, `grape`, `strawberry`, `k_melon`)를 학습시키기 위한 전체 파이프라인이다. Google Colab(GPU 런타임) 환경에서 실행하는 것을 전제로 작성했다.

**전체 흐름**
1. Google Drive에서 데이터셋/배경이미지 압축 해제
2. 클래스별 이미지 수 확인
3. 세부 클래스 기준 7:3 train/val 층화 분할
4. 배경(negative) 이미지를 목표 비율만큼 섞어서 오탐(false positive) 억제
5. `data.yaml` 생성
6. YOLOv8s 기반 전이학습 — **에폭 진행도에 따라 mosaic/mixup 증강 강도를 단계적으로 올리는 커리큘럼 러닝 콜백** 적용
7. 최종 검증 (mAP50 / mAP50-95)

> ⚠️ 참고: 코드 곳곳에 "8개 클래스", "17개 세부 클래스" 등 개발 초기 설계 흔적이 주석으로 남아있다. 최종 모델은 4클래스(`eggplant`/`grape`/`strawberry`/`k_melon`) 기준이며, 학습 결과(`docs/training_results.png`, `docs/confusion_matrix.png`)도 4클래스 기준이다.

**실행 전 준비물**
- Google Drive에 `crop_project/crop_dataset.zip` (클래스별 `images/`, `labels/` 폴더 + `class_map.json` 포함), `crop_project/background_images.zip` 업로드
- Colab 런타임을 GPU로 설정


In [ ]:
import os

# Google Drive가 이미 마운트되어 있는지 확인
if os.path.isdir('/content/drive/MyDrive'):
    print("이미 Google Drive에 연결되어 있습니다.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive 연결 완료.")

# 필요한 라이브러리 설치 (ultralytics: YOLOv8/v11 공식 패키지)
!pip install -U ultralytics

import glob, shutil, random, zipfile
from pathlib import Path

# 경로 설정 (본인 환경에 맞게 수정)
ZIP_PATH = "/content/drive/MyDrive/crop_project/crop_dataset.zip"   # 압축파일 경로
EXTRACT_DIR = "/content/raw_dataset"                    # 압축 해제 위치
WORK_DIR = "/content/yolo_dataset"                      # 학습용으로 재구성될 위치
BACKGROUND_ZIP_PATH = "/content/drive/MyDrive/crop_project/background_images.zip"  # 배경 이미지들이 모여있는 폴더 (라벨 파일은 필요 없음, 이미지만 있으면 됨)
BACKGROUND_EXTRACT_DIR = "/content/background_images"
print(f"압축파일 경로: {ZIP_PATH}")
print(f"압축 해제 위치: {EXTRACT_DIR}")
print(f"학습용으로 재구성될 위치: {WORK_DIR}")

CLASSES = ["eggplant", "grape", "strawberry", "k_melon"]

## 1. 환경 설정 및 데이터셋 압축 해제 준비

Google Drive를 마운트하고 `ultralytics`를 설치한 뒤, 데이터셋/배경이미지 zip 경로와 작업 경로를 지정한다.

In [ ]:
def find_dataset_root(search_dir, classes):
    """classes 폴더들이 모두 들어있는 상위 폴더를 찾아서 반환"""
    for root, dirs, files in os.walk(search_dir):
        if all(cls in dirs for cls in classes):
            return root
    return None

def is_already_extracted(extract_dir, classes):
    root = find_dataset_root(extract_dir, classes)
    if root is None:
        return False, None
    for cls in classes:
        img_dir = os.path.join(root, cls, "images")
        lbl_dir = os.path.join(root, cls, "labels")
        if not os.path.isdir(img_dir) or not os.path.isdir(lbl_dir):
            return False, None
        if len(os.listdir(img_dir)) == 0:
            return False, None
    return True, root

already, dataset_root = is_already_extracted(EXTRACT_DIR, CLASSES)

if already:
    print(f"이미 압축이 해제되어 있습니다. 데이터셋 경로: {dataset_root}")
else:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("압축 해제를 시작합니다...")
    # 파일 경로를 찾을 수 없을 때 발생하는 FileNotFoundError를 진단하기 위한 디버깅 코드 추가
    if not os.path.exists(ZIP_PATH):
        print(f"오류: 압축 파일 '{ZIP_PATH}'을(를) 찾을 수 없습니다.")
        print(f"현재 작업 디렉토리: {os.getcwd()}")
        print(f"'{(Path(ZIP_PATH).parent)}' 디렉토리 내용:")
        try:
            for item in os.listdir(Path(ZIP_PATH).parent):
                print(f"  - {item}")
        except FileNotFoundError:
            print(f"  -> '{(Path(ZIP_PATH).parent)}' 디렉토리가 존재하지 않습니다.")
        except Exception as e:
            print(f"  -> 디렉토리 내용을 읽는 중 오류 발생: {e}")
        # 파일이 없으면 더 이상 진행하지 않고 함수를 종료합니다.
        raise FileNotFoundError(f"압축 파일 '{ZIP_PATH}'을(를) 찾을 수 없습니다.")

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print("압축 해제 완료.")

    dataset_root = find_dataset_root(EXTRACT_DIR, CLASSES)
    if dataset_root is None:
        print("클래스 폴더를 찾지 못했습니다. 압축 해제된 전체 구조:")
        for root, dirs, files in os.walk(EXTRACT_DIR):
            level = root.replace(EXTRACT_DIR, '').count(os.sep)
            indent = '  ' * level
            print(f"{indent}{os.path.basename(root)}/")
        raise FileNotFoundError("crop_dataset 폴더 구조를 확인 후 CLASSES 또는 경로를 다시 설정해주세요.")

print(f"\n사용할 데이터셋 루트: {dataset_root}")
print("\n폴더 구조 확인:")
for cls in CLASSES:
    img_dir = os.path.join(dataset_root, cls, "images")
    lbl_dir = os.path.join(dataset_root, cls, "labels")
    n_img = len(glob.glob(os.path.join(img_dir, "*")))
    n_lbl = len(glob.glob(os.path.join(lbl_dir, "*")))
    print(f"{cls}: images={n_img}, labels={n_lbl}")


# =========================================================================
# ★ 배경 이미지 압축 해제 (crop_dataset과 별개, 클래스 폴더 구조 없이 이미지만 있음)
# =========================================================================
BACKGROUND_ZIP_PATH = "/content/drive/MyDrive/crop_project/background_images.zip"
BACKGROUND_EXTRACT_DIR = "/content/background_images"

def is_background_already_extracted(extract_dir):
    """이미지 파일이 하나라도 이미 있으면 압축 해제된 것으로 간주 (하위 폴더 포함 탐색)"""
    if not os.path.isdir(extract_dir):
        return False
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        if glob.glob(os.path.join(extract_dir, "**", ext), recursive=True):
            return True
    return False

if is_background_already_extracted(BACKGROUND_EXTRACT_DIR):
    print(f"\n배경 이미지가 이미 압축 해제되어 있습니다: {BACKGROUND_EXTRACT_DIR}")
else:
    os.makedirs(BACKGROUND_EXTRACT_DIR, exist_ok=True)
    print("\n배경 이미지 압축 해제를 시작합니다...")

    if not os.path.exists(BACKGROUND_ZIP_PATH):
        print(f"오류: 배경 이미지 압축 파일 '{BACKGROUND_ZIP_PATH}'을(를) 찾을 수 없습니다.")
        print(f"현재 작업 디렉토리: {os.getcwd()}")
        print(f"'{(Path(BACKGROUND_ZIP_PATH).parent)}' 디렉토리 내용:")
        try:
            for item in os.listdir(Path(BACKGROUND_ZIP_PATH).parent):
                print(f"  - {item}")
        except FileNotFoundError:
            print(f"  -> '{(Path(BACKGROUND_ZIP_PATH).parent)}' 디렉토리가 존재하지 않습니다.")
        except Exception as e:
            print(f"  -> 디렉토리 내용을 읽는 중 오류 발생: {e}")
        raise FileNotFoundError(f"배경 이미지 압축 파일 '{BACKGROUND_ZIP_PATH}'을(를) 찾을 수 없습니다.")

    with zipfile.ZipFile(BACKGROUND_ZIP_PATH, 'r') as zf:
        zf.extractall(BACKGROUND_EXTRACT_DIR)
    print("배경 이미지 압축 해제 완료.")

n_bg_images = len(glob.glob(os.path.join(BACKGROUND_EXTRACT_DIR, "**", "*.*"), recursive=True))
print(f"\n배경 이미지 폴더: {BACKGROUND_EXTRACT_DIR}")
print(f"배경 이미지 수: {n_bg_images}")


## 2. 데이터셋 / 배경 이미지 압축 해제

이미 압축이 해제되어 있으면(재실행 시) 다시 풀지 않고 건너뛰도록 idempotent하게 처리했다.
작물 데이터셋(`crop_dataset.zip`)과, 라벨 없이 이미지만 있는 배경 이미지(`background_images.zip`)를 각각 별도 경로에 해제한다.
배경 이미지는 이후 단계에서 "작물이 없는 상황"을 학습시켜 오탐을 줄이는 negative sample로 쓰인다.

In [ ]:
print("클래스별 원본 이미지 개수:")
total = 0;
for cls in CLASSES:
    img_dir = os.path.join(dataset_root, cls, "images")
    n = len(glob.glob(os.path.join(img_dir, "*")))
    print(f"{cls}: {n}장")
    total = total + n
print(f"총 이미지는 {total}장")

## 3. 클래스별 원본 이미지 수 확인

압축 해제된 데이터셋에서 클래스별 이미지 장수를 집계해 데이터 불균형 여부를 먼저 점검한다.

In [ ]:
import os
import glob
import json
import shutil
import random
from pathlib import Path

# ==========================================
# 1. 경로 자동 탐색 및 비율 설정
# ==========================================
random.seed(42)
TRAIN_RATIO = 0.7  # 7:3 비율

# 탐색할 후보 루트 경로들
possible_search_roots = [
    "/content/raw_dataset",
    "/content/crop_raw",
    "/content/crop_dataset",
    "/content/drive/MyDrive"  # 구글 드라이브에 올렸을 경우
]

class_map_path = None

# class_map.json 자동 검색
for search_root in possible_search_roots:
    if os.path.exists(search_root):
        found_files = glob.glob(f"{search_root}/**/class_map.json", recursive=True)
        if found_files:
            class_map_path = found_files[0]
            break

# 여전히 못 찾았다면 현재 디렉토리 전체에서 재검색
if not class_map_path:
    found_files = glob.glob("/content/**/class_map.json", recursive=True)
    if found_files:
        class_map_path = found_files[0]

if not class_map_path or not os.path.exists(class_map_path):
    raise FileNotFoundError("❌ [에러] class_map.json 파일을 찾을 수 없습니다. 경로나 압축 해제 위치를 다시 확인해 주세요.")

print(f"✅ class_map.json 감지 성공! ➔ 경로: {class_map_path}")

# dataset_root는 class_map.json이 위치한 바로 그 폴더로 자동 지정
dataset_root = os.path.dirname(class_map_path)
WORK_DIR = "/content/yolo_dataset"  # 최종 분할 저장될 경로

# ==========================================
# 2. class_map.json 읽기 & 작업 공간 초기화
# ==========================================
with open(class_map_path, "r", encoding="utf-8") as f:
    class_map = json.load(f)

print(f"✅ 총 {len(class_map)}개 세부 클래스 정보 로드 완료!")

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

for split in ["train", "val"]:
    os.makedirs(f"{WORK_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{WORK_DIR}/{split}/labels", exist_ok=True)

# ==========================================
# 3. 17개 세부 클래스별 7:3 균등 층화 분할
# ==========================================
split_summary = {}

for class_name, class_id in class_map.items():
    # 상위 작물 폴더 지정 (eggplant, grape, strawberry, k_melon)
    crop_folder = class_name.split("_")[0]
    if class_name.startswith("k_melon"):
        crop_folder = "k_melon"

    lbl_dir = os.path.join(dataset_root, crop_folder, "labels")
    img_dir = os.path.join(dataset_root, crop_folder, "images")

    matched_stems = []

    if os.path.exists(lbl_dir):
        for txt_file in glob.glob(os.path.join(lbl_dir, "*.txt")):
            with open(txt_file, "r", encoding="utf-8") as tf:
                lines = tf.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) > 0 and int(parts[0]) == class_id:
                        matched_stems.append(Path(txt_file).stem)
                        break

    matched_stems = sorted(list(set(matched_stems)))

    if len(matched_stems) == 0:
        print(f"⚠️ 경고: [{class_name}] (ID: {class_id}) 해당 라벨을 찾지 못했습니다. (경로: {lbl_dir})")
        continue

    stems_copy = matched_stems.copy()
    random.shuffle(stems_copy)

    n_train = round(len(stems_copy) * TRAIN_RATIO)
    train_stems = stems_copy[:n_train]
    val_stems = stems_copy[n_train:]

    def copy_dataset_pairs(stem_list, split_name):
        copied_count = 0
        for stem in stem_list:
            img_path = None
            for ext in [".jpg", ".png", ".jpeg", ".JPG", ".PNG", ".JPEG"]:
                candidate = os.path.join(img_dir, f"{stem}{ext}")
                if os.path.exists(candidate):
                    img_path = candidate
                    break

            src_lbl = os.path.join(lbl_dir, f"{stem}.txt")

            if img_path and os.path.exists(src_lbl):
                dst_img = os.path.join(WORK_DIR, split_name, "images", os.path.basename(img_path))
                dst_lbl = os.path.join(WORK_DIR, split_name, "labels", f"{stem}.txt")

                shutil.copy(img_path, dst_img)
                shutil.copy(src_lbl, dst_lbl)
                copied_count += 1
        return copied_count

    actual_train = copy_dataset_pairs(train_stems, "train")
    actual_val = copy_dataset_pairs(val_stems, "val")

    split_summary[class_name] = {
        "id": class_id,
        "total": len(matched_stems),
        "train": actual_train,
        "val": actual_val,
        "ratio": round(actual_train / len(matched_stems), 2) if len(matched_stems) > 0 else 0
    }

# ==========================================
# 4. 결과 요약 출력
# ==========================================
print("\n🎉 모든 클래스 층화 분할 완료!")
print("-" * 65)
print(f"{'클래스명':<22} {'ID':>4} {'전체':>6} {'Train':>6} {'Val':>6} {'Train비율':>10}")
print("-" * 65)

for cls_name, info in split_summary.items():
    print(f"{cls_name:<22} {info['id']:>4} {info['total']:>6} {info['train']:>6} {info['val']:>6} {info['ratio']:>10.2f}")

print("-" * 65)
print(f"📦 최종 Train 이미지 수: {len(glob.glob(WORK_DIR + '/train/images/*'))}장")
print(f"📦 최종 Val 이미지 수  : {len(glob.glob(WORK_DIR + '/val/images/*'))}장")

## 4. Train / Val 층화 분할 (7:3)

`class_map.json`을 자동으로 탐색해 세부 클래스별 라벨을 스캔하고, 클래스별로 균등하게 7:3 비율로 train/val을 분할해 복사한다.
전체를 한 번에 랜덤 분할하지 않고 **클래스 단위로 층화 분할(stratified split)**하는 이유는, 클래스 간 이미지 수 편차가 커서(예: strawberry 4,502장 vs k_melon 640장) 단순 랜덤 분할 시 특정 클래스가 val에 과소/과대 포함될 수 있기 때문이다.

`random.seed(42)`로 고정해 재현 가능하게 했다.

In [ ]:
import os
import glob
import shutil
import random
from pathlib import Path
##BACKGROUND_EXTRACT_DIR
train_ratio = 0.1 # 배경 이미지가 "최종 전체 데이터셋"에서 차지할 목표 비율 (0.05~0.10 권장)
random.seed(42)

# =========================================================================
# 1. 현재 train/val에 이미 있는 이미지 수 세기 (목표 배경 장수 계산용)
# =========================================================================
def count_existing_images(work_dir):
    total = 0
    for split in ("train", "val"):
        img_dir = os.path.join(work_dir, split, "images")
        if os.path.isdir(img_dir):
            total += len(glob.glob(os.path.join(img_dir, "*")))
    return total


# =========================================================================
# 3. 목표 배경 이미지 장수 계산
#    target_ratio = 배경장수 / (배경장수 + 기존장수)  =>  배경장수 = 기존장수 * ratio / (1-ratio)
# =========================================================================
def calc_target_background_count(existing_count, target_ratio):
    if target_ratio >= 1.0:
        raise ValueError("target_ratio는 1.0 미만이어야 합니다.")
    return round(existing_count * target_ratio / (1 - target_ratio))


# =========================================================================
# 4. 배경 이미지 수집 + 샘플링
# =========================================================================
def collect_background_images(background_dir, target_count, seed):
    img_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        img_paths.extend(glob.glob(os.path.join(background_dir, "**", ext), recursive=True))
    img_paths = sorted(set(img_paths))

    random.seed(seed)
    random.shuffle(img_paths)

    if len(img_paths) < target_count:
        print(f"[경고] 목표 {target_count}장인데 실제 배경 이미지가 {len(img_paths)}장뿐입니다. "
              f"있는 만큼만 사용합니다.")
        return img_paths

    return img_paths[:target_count]


# =========================================================================
# 5. train/val에 빈 라벨과 함께 병합
# =========================================================================
def merge_background_images(selected_paths):
    random.shuffle(selected_paths)
    n_train = round(len(selected_paths) * train_ratio)
    train_paths = selected_paths[:n_train]
    val_paths = selected_paths[n_train:]

    def copy_set(paths, split_name):
        count = 0
        for img_path in paths:
            stem = Path(img_path).stem
            ext = Path(img_path).suffix
            new_name = f"background_{stem}"

            dst_img = os.path.join(WORK_DIR, split_name, "images", f"{new_name}{ext}")
            dst_lbl = os.path.join(WORK_DIR, split_name, "labels", f"{new_name}.txt")

            os.makedirs(os.path.dirname(dst_img), exist_ok=True)
            os.makedirs(os.path.dirname(dst_lbl), exist_ok=True)

            shutil.copy(img_path, dst_img)
            open(dst_lbl, "w").close()   # ★ 빈 라벨 파일 (class_id 없음)

            count += 1
        return count

    n_train_actual = copy_set(train_paths, "train")
    n_val_actual = copy_set(val_paths, "val")

    return n_train_actual, n_val_actual


# =========================================================================
# 6. 실행
# =========================================================================

print("[1/4] 기존 train/val 이미지 수 확인 중...")
existing_count = count_existing_images(WORK_DIR)
print(f"  기존 이미지 수: {existing_count}")

print("\n[2/4] 목표 배경 이미지 장수 계산 중...")
target_count = calc_target_background_count(existing_count, train_ratio)
print(f"  목표 비율 {train_ratio*100:.0f}% -> 배경 이미지 {target_count}장 필요")

print("\n[3/4] 배경 이미지 수집 및 샘플링 중...")
selected_paths = collect_background_images(BACKGROUND_EXTRACT_DIR, target_count, '42')
print(f"  선택된 배경 이미지: {len(selected_paths)}장")

print("\n[4/4] train/val에 병합 중 (빈 라벨 생성)...")
n_train, n_val = merge_background_images(selected_paths)
print(f"  train에 {n_train}장, val에 {n_val}장 추가됨 (전부 빈 라벨, class_id 없음)")

final_total = existing_count + n_train + n_val
actual_ratio = (n_train + n_val) / final_total if final_total else 0
print(f"\n완료. 최종 배경 이미지 비율: 약 {actual_ratio*100:.1f}%")

## 5. 배경(negative) 이미지 비율 맞춰 병합

작물 이미지만으로 학습하면 "작물이 아예 없는 화면"에서도 뭔가를 탐지하려는 오탐이 발생하기 쉽다. 이를 줄이기 위해 배경 이미지를 최종 데이터셋의 약 5~10% 비율이 되도록 계산해서 샘플링하고, **라벨 파일은 비워둔 채(class_id 없이)** train/val에 병합한다.

In [ ]:
# data.yaml 생성 (YOLO 학습 설정 파일)
yaml_content = f"""
path: {WORK_DIR}
train: train/images
val: val/images

nc: 4
names:
  0 : eggplant,
  1 : grape,
  2 : strawberry,
  3 : k_melon
"""

with open(f"{WORK_DIR}/data.yaml", "w") as f:
    f.write(yaml_content.strip())


print(yaml_content)

## 6. `data.yaml` 생성

최종 4클래스(`eggplant`/`grape`/`strawberry`/`k_melon`) 기준으로 YOLO 학습에 필요한 `data.yaml`을 생성한다.

In [ ]:
import os
from ultralytics import YOLO


# =========================================================================
# 1. 설정
# =========================================================================
CONFIG = {
    # colab_split_dataset.py가 생성한 data.yaml 경로
    "data_yaml_path": "/content/yolo_dataset/data.yaml",

    # 사전학습 가중치 (전이학습 시작점)
    "base_model": "yolov8s.pt",   # 클래스 8개 + 병해충까지 구분해야 하므로 n보다 s 권장

    "epochs": 80,
    "imgsz": 640,
    "batch": 16,
    "patience": 20,          # 성능 개선 없으면 20 epoch 후 조기 종료
    # "lr0": 0.001,          # 초기 학습률을 직접 지정하고 싶으면 주석 해제 (기본값은 YOLO 기본 하이퍼파라미터 사용)

    "project": "/content/drive/MyDrive/crop_project/",  # 결과를 드라이브에 저장
    "name": "curriculum_crop_detect_v1",
    "device": 0,              # GPU 사용 (런타임 > GPU로 변경 필요)

    # --- 커리큘럼 러닝 구간별 목표 증강 강도 ---
    # get_curriculum_params()의 아이디어를 그대로 반영:
    #   진행도 0~30%  : mosaic를 0에서 서서히 증가, mixup은 아직 0
    #   진행도 30~70% : mosaic/mixup 모두 증가
    #   진행도 70~100%: 최대 강도로 고정
    "curriculum": {
        "phase1_end": 0.3,
        "phase2_end": 0.7,
        "mosaic_phase2_start": 0.3,
        "mixup_phase3": 0.5,
        "mosaic_phase3": 0.8,
    },
}


# =========================================================================
# 2. 커리큘럼 러닝 콜백
#    - Ultralytics의 trainer 객체를 받아서, 매 epoch 시작 시 augmentation
#      하이퍼파라미터(trainer.args.mosaic, trainer.args.mixup)를 갱신한다.
#    - 학습 루프/손실 계산에는 전혀 손대지 않으므로 Ultralytics 엔진의
#      안정성을 그대로 유지한다.
# =========================================================================
def make_curriculum_callback(curriculum_cfg):
    def on_train_epoch_start(trainer):
        progress = trainer.epoch / trainer.epochs

        phase1_end = curriculum_cfg["phase1_end"]
        phase2_end = curriculum_cfg["phase2_end"]

        if progress < phase1_end:
            # 1단계: mosaic만 0 -> phase1_end 지점까지 서서히 증가, mixup은 아직 사용 안 함
            mosaic = progress / phase1_end
            mixup = 0.0
        elif progress < phase2_end:
            # 2단계: mosaic/mixup 모두 증가
            span = phase2_end - phase1_end
            local_progress = (progress - phase1_end) / span
            mosaic = curriculum_cfg["mosaic_phase2_start"] + local_progress * (
                curriculum_cfg["mosaic_phase3"] - curriculum_cfg["mosaic_phase2_start"]
            )
            mixup = local_progress * curriculum_cfg["mixup_phase3"]
        else:
            # 3단계: 최대 강도로 고정
            mosaic = curriculum_cfg["mosaic_phase3"]
            mixup = curriculum_cfg["mixup_phase3"]

        trainer.args.mosaic = round(mosaic, 3)
        trainer.args.mixup = round(mixup, 3)

        print(f"[커리큘럼] epoch {trainer.epoch}/{trainer.epochs} "
              f"(진행도 {progress:.2f}) -> mosaic={trainer.args.mosaic}, mixup={trainer.args.mixup}")

    return on_train_epoch_start


# =========================================================================
# 3. 실행
# =========================================================================
def run_training(config=None):
    cfg = config or CONFIG

    print("[초기화] 모델 로드 중...")
    model = YOLO(cfg["base_model"])

    print("[초기화] 커리큘럼 러닝 콜백 등록 중...")
    callback = make_curriculum_callback(cfg["curriculum"])
    model.add_callback("on_train_epoch_start", callback)

    print("[학습 시작]")
    train_kwargs = dict(
        data=cfg["data_yaml_path"],
        epochs=cfg["epochs"],
        imgsz=cfg["imgsz"],
        batch=cfg["batch"],
        patience=cfg["patience"],
        project=cfg["project"],
        name=cfg["name"],
        device=cfg["device"],
        # 초기값은 커리큘럼 콜백이 매 epoch마다 덮어쓰므로 0으로 시작
        mosaic=0.0,
        mixup=0.0,
    )
    if "lr0" in cfg:
        train_kwargs["lr0"] = cfg["lr0"]

    results = model.train(**train_kwargs)

    print("\n[학습 완료] 검증 성능 확인 중...")
    metrics = model.val()
    print(f"mAP50-95: {metrics.box.map:.4f}")
    print(f"mAP50: {metrics.box.map50:.4f}")

    return model, results, metrics


if __name__ == "__main__":
    run_training(CONFIG)

## 7. 커리큘럼 러닝 기반 학습

`yolov8s.pt`를 백본으로 전이학습을 진행한다. 이 프로젝트에서 직접 구현한 핵심 부분은 **`on_train_epoch_start` 콜백을 이용한 커리큘럼 러닝**이다.

- 학습 진행도(`epoch / epochs`)를 0~1 구간으로 놓고 3단계로 나눔
  - **0~30%**: mosaic 증강을 0에서 서서히 증가, mixup은 아직 적용 안 함 (쉬운 샘플부터 학습)
  - **30~70%**: mosaic/mixup 둘 다 점진적으로 증가
  - **70~100%**: 최대 강도로 고정 (가장 어려운/다양한 조합의 증강 데이터로 마무리)
- Ultralytics의 학습 루프 자체는 건드리지 않고 `trainer.args.mosaic`/`trainer.args.mixup` 값만 매 epoch 시작 시 갱신하는 방식이라, 엔진 안정성을 유지하면서 증강 스케줄만 커스터마이징했다.
- `patience=20`으로 조기 종료를 걸어 불필요한 학습을 방지했다. (`docs/training_results.png`에서 mAP50이 50~60 epoch 이후 평탄화되는 것도 이와 관련)

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/crop_project/curriculum_crop_detect_v1/weights/best.pt")
metrics = model.val(
    project="/content/drive/MyDrive/crop_project/",
    name="crop_detect_v1_val" # Using a different name for val results to avoid conflict with train results
)
print(metrics.box.map)    # mAP50-95
print(metrics.box.map50)  # mAP50